# Expense Classification using NLP
## AI/ML/NLP Candidate Assessment

This notebook performs end-to-end analysis: data loading, cleaning, EDA, and building a text classification model to categorize expenses into **Services**, **Equipment**, or **Material** based on the remarks text.

In [ ]:
# Install required libraries
!pip install pandas numpy matplotlib seaborn scikit-learn openpyxl optuna xgboost lightgbm wordcloud --quiet
import warnings
warnings.filterwarnings('ignore')

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from collections import Counter

# sklearn imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print("All libraries imported successfully")

## 2. Data Loading and Basic EDA

In [ ]:
# Load the dataset
df = pd.read_excel('data.xlsx')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(10)

In [ ]:
# Last few rows
df.tail()

In [ ]:
# Basic info
print("Shape:", df.shape)
print("\nColumn Names:", list(df.columns))
print("\nData Types:")
print(df.dtypes)

In [ ]:
# Missing values
print("Missing Values:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Duplicates check
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate remarks: {df['Remarks'].duplicated().sum()}")

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Unique values per column
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

### Observations from Basic EDA
- All entries are from FY23 (single financial year)
- Credit column is always 0 so Net equals Debit
- The Remarks column contains the text we need for classification
- Some remarks are empty or very short which will need handling

## 3. Data Cleaning and Target Label Creation

### Creating Target Labels

The task requires classifying expenses into: **Services**, **Equipment**, or **Material**

Since no explicit labels exist in the data we need to create them using domain knowledge:
- **Services**: Consultancy, installation work, maintenance, commissioning, civil work, provisions
- **Equipment**: Physical devices/machines - AC units, cameras, dispensers, chairs, racks, projectors
- **Material**: Raw materials and consumables - pipes, cables, wires, insulation, fittings

In [ ]:
# Clean the data first
df_clean = df.copy()

# Drop rows with empty remarks
df_clean = df_clean[df_clean['Remarks'].notna() & (df_clean['Remarks'].str.strip() != '')]
print(f"After removing empty remarks: {df_clean.shape[0]} rows")

# Convert Debit to numeric
df_clean['Debit'] = pd.to_numeric(df_clean['Debit'], errors='coerce')
df_clean['Credit'] = pd.to_numeric(df_clean['Credit'], errors='coerce')
df_clean['Net'] = pd.to_numeric(df_clean['Net'], errors='coerce')

# Fill any remaining NaN in numeric columns
df_clean['Debit'].fillna(0, inplace=True)
df_clean['Net'].fillna(0, inplace=True)

print(f"Final clean dataset: {df_clean.shape}")
df_clean.head()

In [ ]:
# Create target labels using keyword-based rules
# This is our labeling strategy based on domain understanding of the remarks

def classify_expense(text):
    text_lower = str(text).lower().strip()
    
    # Services indicators
    services_patterns = [
        r'^services\b',
        r'consultancy', r'consulting', r'commissioning engineers',
        r'provision\b', r'exp reclass', r'exp transfer', r'expense provision',
        r'interior and mep work', r'mep.*hoto', r'hoto',
        r're-furnishment', r'upgradation', r'civil work', r'carpentry',
        r'installation charges', r'wiring and installation',
        r'dashboard maintenance', r'web portal access',
        r'software develop', r'customiz', r'dismantling',
        r'charges for.*loading', r'freight$',
        r'trf to exp', r'space matrix cost',
        r'fire fighting'
    ]
    
    # Equipment indicators  
    equipment_patterns = [
        r'dispenser', r'almirah', r'split ac\b', r'\bac\b.*star',
        r'camera\b', r'cctv', r'projector', r'\btv\b', r'led tv', r'uhd tv',
        r'exhaust fan', r'\bchair\b', r'locker', r'water cooler', r'pump\b',
        r'wheel chair', r'\bups\b', r'tester\b', r'hard drive', r'hard disk',
        r'\bswitch\b.*port', r'shoe shine', r'air blower', r'extinguisher',
        r'pallet', r'pos machine', r'kiosk', r'scent diffuser',
        r'hand dryer', r'paper dispenser', r'music system', r'\brack\b',
        r'seesaw', r'swing\b', r'\bslide\b', r'\bphone\b', r'air cleaner',
        r'abt meter', r'eye wash', r'\bmachine\b', r'nvr\b', r'recorder',
        r'baby diaper', r'sanitary pad', r'air quality monitoring sensor',
        r'android media player', r'evacuation chair', r'\bpbx\b',
        r'music device'
    ]
    
    # Material indicators
    material_patterns = [
        r'copper piping', r'\bpipe\b', r'\bcable\b(?!.*tray)',
        r'conduit', r'\belbow\b', r'nut bolts', r'insul[la]+tion\b',
        r'\bangel\b', r'\brod\b', r'\bsfp module\b', r'patch cord',
        r'jumper wire', r'fragrance oil', r'glass plate', r'power adapter',
        r'crimping tool', r'pigtail', r'wiring.*mm', r'ms stand',
        r'\bdrain\b.*pipe', r'electrical wiring'
    ]
    
    # Check services first (explicit keyword at start is strongest signal)
    if re.match(r'^services\b', text_lower):
        return 'Services'
    
    for pattern in services_patterns:
        if re.search(pattern, text_lower):
            return 'Services'
    
    for pattern in equipment_patterns:
        if re.search(pattern, text_lower):
            return 'Equipment'
    
    for pattern in material_patterns:
        if re.search(pattern, text_lower):
            return 'Material'
    
    # SITC usually means equipment procurement (Supply Installation Testing Commissioning)
    if 'sitc' in text_lower or 'supply' in text_lower:
        # Check if its more services-like
        if any(w in text_lower for w in ['charges', 'maintenance', 'access']):
            return 'Services'
        return 'Equipment'
    
    # Remaining with installation/work/fixing keywords -> Services
    if any(w in text_lower for w in ['installation', 'work', 'fixing', 'supplying']):
        return 'Services'
    
    # Default based on amount and remaining text patterns
    if any(w in text_lower for w in ['ahuja', 'narema', 'euronics', 'global engineering', 'aaztec', 'pinnacle']):
        return 'Equipment'
    
    return 'Equipment'  # default for ambiguous procurement items

df_clean['Category'] = df_clean['Remarks'].apply(classify_expense)
print("Target Distribution:")
print(df_clean['Category'].value_counts())
print(f"\nTotal classified: {len(df_clean)}")

## 4. Data Visualization

In [ ]:
# Plot 1: Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot
df_clean['Category'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#3498db', '#e74c3c'])
axes[0].set_title('Expense Category Distribution')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
df_clean['Category'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                          colors=['#2ecc71', '#3498db', '#e74c3c'])
axes[1].set_title('Category Proportions')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

**Observation:** Equipment is the largest category followed by Services then Material. The dataset is imbalanced which we need to account for during model evaluation.

In [ ]:
# Plot 2: Debit amount distribution by category
fig, ax = plt.subplots(figsize=(10, 6))
for cat in df_clean['Category'].unique():
    subset = df_clean[df_clean['Category'] == cat]['Debit']
    # clip to remove extreme outliers for visualization
    subset_clipped = subset[subset < subset.quantile(0.95)]
    ax.hist(subset_clipped, alpha=0.6, label=cat, bins=30)

ax.set_title('Distribution of Debit Amount by Category (95th percentile)')
ax.set_xlabel('Debit Amount (INR)')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** Services tend to have higher debit amounts (large project costs) while Material items are generally lower value. Equipment spans a wide range.

In [ ]:
# Plot 3: Box plot of debit by category
fig, ax = plt.subplots(figsize=(10, 6))
df_plot = df_clean[df_clean['Debit'] < df_clean['Debit'].quantile(0.95)]
sns.boxplot(data=df_plot, x='Category', y='Debit', ax=ax)
ax.set_title('Debit Amount Distribution by Category')
ax.set_xlabel('Category')
ax.set_ylabel('Debit Amount (INR)')
plt.tight_layout()
plt.show()

**Observation:** Services has the highest median spend. Material items cluster at lower amounts. There are significant outliers in all categories.

In [ ]:
# Plot 4: Text length distribution
df_clean['text_length'] = df_clean['Remarks'].str.len()

fig, ax = plt.subplots(figsize=(10, 6))
for cat in df_clean['Category'].unique():
    subset = df_clean[df_clean['Category'] == cat]['text_length']
    ax.hist(subset, alpha=0.6, label=cat, bins=25)

ax.set_title('Remarks Text Length by Category')
ax.set_xlabel('Character Count')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** Services remarks tend to be longer (more descriptive text) while Material and Equipment have shorter descriptions. Text length could be a useful feature.

In [ ]:
# Plot 5: Word count distribution
df_clean['word_count'] = df_clean['Remarks'].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=df_clean, x='Category', y='word_count', ax=ax)
ax.set_title('Word Count Distribution by Category')
ax.set_xlabel('Category')
ax.set_ylabel('Number of Words')
plt.tight_layout()
plt.show()

**Observation:** Services descriptions are wordier on average. This makes sense as service contracts tend to have more detailed descriptions.

In [ ]:
# Plot 6: Top words per category
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
categories = ['Services', 'Equipment', 'Material']

for i, cat in enumerate(categories):
    text = ' '.join(df_clean[df_clean['Category'] == cat]['Remarks'].str.lower())
    # remove common stopwords and numbers
    stopwords = {'the', 'of', 'and', 'for', 'in', 'with', 'to', 'as', 'a', 'at', 'on', 'is', 
                 'per', 'no', 'dt', 'from', 'by', 'or', 'an', 'be', '-', '&', 'its'}
    words = [w for w in re.findall(r'[a-z]+', text) if w not in stopwords and len(w) > 2]
    top_words = Counter(words).most_common(15)
    
    words_list = [w[0] for w in top_words]
    counts = [w[1] for w in top_words]
    
    axes[i].barh(words_list, counts, color=sns.color_palette('Set2')[i])
    axes[i].set_title(f'Top Words - {cat}')
    axes[i].invert_yaxis()

plt.tight_layout()
plt.show()

**Observation:** Each category has distinct vocabulary. Services shows words like 'commissioning' 'consultancy' 'work'. Equipment shows 'camera' 'dispenser' 'chair'. Material shows 'pipe' 'cable' 'insulation'. This confirms text-based classification is viable.

In [ ]:
# Plot 7: Debit statistics per category
stats = df_clean.groupby('Category')['Debit'].agg(['mean', 'median', 'sum'])
stats_normalized = stats / stats.max()

fig, ax = plt.subplots(figsize=(10, 6))
stats_normalized.plot(kind='bar', ax=ax)
ax.set_title('Normalized Debit Statistics by Category')
ax.set_xlabel('Category')
ax.set_ylabel('Normalized Value')
ax.legend(['Mean', 'Median', 'Total Sum'])
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print("Actual values:")
print(stats.round(2))

**Observation:** Services dominate in total spending due to large project costs. Equipment has the highest count but moderate individual amounts.

In [ ]:
# Plot 8: Correlation between numeric features
df_numeric = df_clean[['Debit', 'text_length', 'word_count']].copy()
# Add category as numeric for correlation
le_temp = LabelEncoder()
df_numeric['Category_encoded'] = le_temp.fit_transform(df_clean['Category'])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df_numeric.corr(), annot=True, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()

**Observation:** Text length and word count are highly correlated (expected). Debit amount has weak correlation with category suggesting text features will be more important than amount alone.

In [ ]:
# Plot 9: Scatter plot - text length vs debit colored by category
fig, ax = plt.subplots(figsize=(10, 6))
for cat in categories:
    subset = df_clean[df_clean['Category'] == cat]
    ax.scatter(subset['text_length'], np.log1p(subset['Debit']), 
               alpha=0.6, label=cat, s=50)

ax.set_title('Text Length vs Log(Debit) by Category')
ax.set_xlabel('Text Length (characters)')
ax.set_ylabel('Log(Debit Amount)')
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** Categories show some clustering in the text-length vs amount space but with significant overlap. Pure numeric features wont be enough - we need NLP on the text.

In [ ]:
# Plot 10: Cumulative spending distribution
fig, ax = plt.subplots(figsize=(10, 6))
for cat in categories:
    subset = df_clean[df_clean['Category'] == cat]['Debit'].sort_values()
    cumulative = subset.cumsum() / subset.sum()
    ax.plot(range(len(cumulative)), cumulative.values, label=cat, linewidth=2)

ax.set_title('Cumulative Spending Distribution by Category')
ax.set_xlabel('Number of Transactions (sorted by amount)')
ax.set_ylabel('Cumulative Proportion of Total Spend')
ax.legend()
plt.tight_layout()
plt.show()

**Observation:** Services spending is dominated by a few large transactions (steep curve early). Equipment spending is more evenly distributed across transactions.

## 5. Data Preprocessing

In [ ]:
# Text preprocessing function
def preprocess_text(text):
    text = str(text).lower()
    # remove numeric codes at the start (like asset IDs)
    text = re.sub(r'^\d{6,}\s*', '', text)
    # remove dates
    text = re.sub(r'\d{2}/\d{2}/\d{2,4}', '', text)
    # remove PO numbers
    text = re.sub(r'po\s*\d+', '', text)
    # remove special characters but keep spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    # remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean['text_processed'] = df_clean['Remarks'].apply(preprocess_text)
print("Sample preprocessed text:")
for i in range(5):
    print(f"  Original: {df_clean['Remarks'].iloc[i][:80]}")
    print(f"  Cleaned:  {df_clean['text_processed'].iloc[i][:80]}")
    print()

In [ ]:
# Prepare features and target
X_text = df_clean['text_processed']
y = df_clean['Category']

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Classes:", le.classes_)
print("Distribution:", dict(zip(le.classes_, np.bincount(y_encoded))))

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),  # unigrams and bigrams
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(X_text)
print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

# Add numeric features
X_numeric = df_clean[['Debit', 'text_length', 'word_count']].values
from scipy.sparse import hstack
X_combined = hstack([X_tfidf, X_numeric])
print(f"Combined feature matrix: {X_combined.shape}")

In [ ]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTrain distribution: {dict(zip(le.classes_, np.bincount(y_train)))}")
print(f"Test distribution: {dict(zip(le.classes_, np.bincount(y_test)))}")

## 6. Model Building

### Approach Comparison

**Approach 1: Traditional ML with TF-IDF Features**
- Use TF-IDF to vectorize text and train classical ML classifiers
- Advantages: Fast training, interpretable, works well with small datasets, no GPU needed
- Disadvantages: Cannot capture semantic meaning, limited context understanding

**Approach 2: Transformer-based (BERT/DistilBERT)**  
- Fine-tune a pre-trained language model for classification
- Advantages: Captures semantic meaning, handles context well, state-of-the-art NLP
- Disadvantages: Requires GPU, large model for small dataset, prone to overfitting with ~268 samples

**Choice: Approach 1 (TF-IDF + Classical ML)**

Rationale: With only ~268 samples transformer models would severely overfit. The text contains strong keyword signals (domain-specific vocabulary) that TF-IDF captures well. Classical models with proper tuning will generalize better here.

In [ ]:
# Model 1: Logistic Regression (baseline)
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, lr_pred):.4f}")
print(f"Macro F1: {f1_score(y_test, lr_pred, average='macro'):.4f}")

In [ ]:
# Model 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(f"Macro F1: {f1_score(y_test, rf_pred, average='macro'):.4f}")

In [ ]:
# Model 3: XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=42, eval_metric='mlogloss', verbosity=0
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
print(f"XGBoost Accuracy: {accuracy_score(y_test, xgb_pred):.4f}")
print(f"Macro F1: {f1_score(y_test, xgb_pred, average='macro'):.4f}")

In [ ]:
# Model 4: LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=42, class_weight='balanced', verbose=-1
)
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)
print(f"LightGBM Accuracy: {accuracy_score(y_test, lgb_pred):.4f}")
print(f"Macro F1: {f1_score(y_test, lgb_pred, average='macro'):.4f}")

In [ ]:
# Model 5: Extra Trees
et_model = ExtraTreesClassifier(n_estimators=200, random_state=42, class_weight='balanced')
et_model.fit(X_train, y_train)
et_pred = et_model.predict(X_test)
print(f"Extra Trees Accuracy: {accuracy_score(y_test, et_pred):.4f}")
print(f"Macro F1: {f1_score(y_test, et_pred, average='macro'):.4f}")

## 7. Hyperparameter Optimization with Optuna

In [ ]:
# Optuna tuning for Logistic Regression
def lr_objective(trial):
    C = trial.suggest_float('C', 0.01, 100, log=True)
    solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear'])
    
    model = LogisticRegression(C=C, solver=solver, max_iter=1000, 
                                random_state=42, class_weight='balanced')
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro')
    return scores.mean()

study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(lr_objective, n_trials=30, show_progress_bar=False)
print(f"Best LR params: {study_lr.best_params}")
print(f"Best LR F1: {study_lr.best_value:.4f}")

In [ ]:
# Optuna tuning for Random Forest
def rf_objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)
    
    model = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth,
        min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
        random_state=42, class_weight='balanced'
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro')
    return scores.mean()

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(rf_objective, n_trials=30, show_progress_bar=False)
print(f"Best RF params: {study_rf.best_params}")
print(f"Best RF F1: {study_rf.best_value:.4f}")

In [ ]:
# Optuna tuning for XGBoost
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
    }
    
    model = xgb.XGBClassifier(**params, random_state=42, eval_metric='mlogloss', verbosity=0)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro')
    return scores.mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(xgb_objective, n_trials=30, show_progress_bar=False)
print(f"Best XGB params: {study_xgb.best_params}")
print(f"Best XGB F1: {study_xgb.best_value:.4f}")

In [ ]:
# Optuna tuning for LightGBM
def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 60),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
    }
    
    model = lgb.LGBMClassifier(**params, random_state=42, class_weight='balanced', verbose=-1)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro')
    return scores.mean()

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(lgb_objective, n_trials=30, show_progress_bar=False)
print(f"Best LGB params: {study_lgb.best_params}")
print(f"Best LGB F1: {study_lgb.best_value:.4f}")

In [ ]:
# Optuna tuning for Extra Trees
def et_objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)
    
    model = ExtraTreesClassifier(
        n_estimators=n_estimators, max_depth=max_depth,
        min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
        random_state=42, class_weight='balanced'
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro')
    return scores.mean()

study_et = optuna.create_study(direction='maximize')
study_et.optimize(et_objective, n_trials=30, show_progress_bar=False)
print(f"Best ET params: {study_et.best_params}")
print(f"Best ET F1: {study_et.best_value:.4f}")

## 8. Cross Validation with Tuned Models

In [ ]:
# Train tuned models
best_lr = LogisticRegression(**study_lr.best_params, max_iter=1000, 
                             random_state=42, class_weight='balanced')
best_rf = RandomForestClassifier(**study_rf.best_params, random_state=42, class_weight='balanced')
best_xgb = xgb.XGBClassifier(**study_xgb.best_params, random_state=42, 
                              eval_metric='mlogloss', verbosity=0)
best_lgb = lgb.LGBMClassifier(**study_lgb.best_params, random_state=42, 
                               class_weight='balanced', verbose=-1)
best_et = ExtraTreesClassifier(**study_et.best_params, random_state=42, class_weight='balanced')

models = {
    'Logistic Regression': best_lr,
    'Random Forest': best_rf,
    'XGBoost': best_xgb,
    'LightGBM': best_lgb,
    'Extra Trees': best_et
}

# Cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro')
    cv_results[name] = {
        'mean_f1': scores.mean(),
        'std_f1': scores.std(),
        'scores': scores
    }
    print(f"{name:25s} | Mean F1: {scores.mean():.4f} +/- {scores.std():.4f}")

## 9. Model Comparison

In [ ]:
# Fit all models on training data and evaluate on test set
comparison_data = []

for name, model in models.items():
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    comparison_data.append({
        'Model': name,
        'CV F1 (mean)': cv_results[name]['mean_f1'],
        'CV F1 (std)': cv_results[name]['std_f1'],
        'Train Accuracy': accuracy_score(y_train, train_pred),
        'Test Accuracy': accuracy_score(y_test, test_pred),
        'Test F1 (macro)': f1_score(y_test, test_pred, average='macro')
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Test F1 (macro)', ascending=False)
comparison_df = comparison_df.round(4)
print(comparison_df.to_string(index=False))

In [ ]:
# Identify best model
best_model_name = comparison_df.iloc[0]['Model']
print(f"\nBest model: {best_model_name}")
print(f"Test F1 (macro): {comparison_df.iloc[0]['Test F1 (macro)']:.4f}")
print(f"Test Accuracy: {comparison_df.iloc[0]['Test Accuracy']:.4f}")

# Get the best fitted model
best_model = models[best_model_name]

## 10. Predictions

In [ ]:
# Generate predictions on test set using best model
final_predictions = best_model.predict(X_test)
final_pred_labels = le.inverse_transform(final_predictions)

# Create prediction dataframe
pred_df = pd.DataFrame({
    'Actual': le.inverse_transform(y_test),
    'Predicted': final_pred_labels
})

# Show sample predictions
print("Sample Predictions (first 20):")
print(pred_df.head(20).to_string())
print(f"\nTotal correct: {(pred_df['Actual'] == pred_df['Predicted']).sum()}/{len(pred_df)}")

## 11. Model Evaluation

### Metric Justification

We use **Macro F1-Score** and **Weighted F1-Score** as our primary metrics because:

1. **Macro F1**: Treats all classes equally regardless of support. Important because Material class has fewer samples and we want good performance across all categories.

2. **Weighted F1**: Accounts for class imbalance by weighting each class by its proportion. Gives a realistic overall performance picture.

We also report accuracy and per-class precision/recall for completeness.

In [ ]:
# Classification Report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, final_predictions, target_names=le.classes_))

# Key metrics
acc = accuracy_score(y_test, final_predictions)
f1_macro = f1_score(y_test, final_predictions, average='macro')
f1_weighted = f1_score(y_test, final_predictions, average='weighted')
prec_macro = precision_score(y_test, final_predictions, average='macro')
rec_macro = recall_score(y_test, final_predictions, average='macro')

print(f"\nOverall Accuracy: {acc:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")
print(f"Weighted F1 Score: {f1_weighted:.4f}")
print(f"Macro Precision: {prec_macro:.4f}")
print(f"Macro Recall: {rec_macro:.4f}")

## 12. Evaluation Visualizations

In [ ]:
# Plot 1: Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, final_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title(f'Confusion Matrix - {best_model_name}')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

# Print misclassification details
print("\nMisclassified samples:")
misclassified = pred_df[pred_df['Actual'] != pred_df['Predicted']]
print(f"Total misclassified: {len(misclassified)} out of {len(pred_df)}")

In [ ]:
# Plot 2: Per-class F1 scores
f1_per_class = f1_score(y_test, final_predictions, average=None)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(le.classes_, f1_per_class, color=['#2ecc71', '#3498db', '#e74c3c'])
ax.set_title('F1 Score by Category')
ax.set_xlabel('Category')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.1)
for bar, val in zip(bars, f1_per_class):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.3f}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Model comparison chart
fig, ax = plt.subplots(figsize=(10, 6))
models_names = comparison_df['Model'].values
test_f1 = comparison_df['Test F1 (macro)'].values
cv_f1 = comparison_df['CV F1 (mean)'].values

x = np.arange(len(models_names))
width = 0.35

bars1 = ax.bar(x - width/2, cv_f1, width, label='CV F1 (macro)', color='#3498db')
bars2 = ax.bar(x + width/2, test_f1, width, label='Test F1 (macro)', color='#e74c3c')

ax.set_xlabel('Model')
ax.set_ylabel('F1 Score (Macro)')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models_names, rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4: Feature importance (if tree-based model)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    
    # Get feature names
    tfidf_features = tfidf.get_feature_names_out()
    numeric_features = ['Debit', 'text_length', 'word_count']
    all_features = list(tfidf_features) + numeric_features
    
    # Top 20 features
    top_idx = np.argsort(importances)[-20:]
    top_features = [all_features[i] for i in top_idx]
    top_importances = importances[top_idx]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(top_features, top_importances, color='#3498db')
    ax.set_title(f'Top 20 Feature Importances - {best_model_name}')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    # For logistic regression show coefficients
    print("Feature importance via model coefficients:")
    tfidf_features = tfidf.get_feature_names_out()
    numeric_features = ['Debit', 'text_length', 'word_count']
    all_features = list(tfidf_features) + numeric_features
    
    coef_abs = np.abs(best_model.coef_).mean(axis=0)
    top_idx = np.argsort(coef_abs)[-20:]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh([all_features[i] for i in top_idx], coef_abs[top_idx], color='#3498db')
    ax.set_title(f'Top 20 Features by Coefficient Magnitude - {best_model_name}')
    ax.set_xlabel('Mean Absolute Coefficient')
    plt.tight_layout()
    plt.show()

**Observation:** The most important features are domain-specific keywords like 'services' 'commissioning' 'pipe' 'cable' 'camera' etc. This validates that TF-IDF captures the discriminative vocabulary effectively.

In [ ]:
# Plot 5: Cross-validation scores distribution
fig, ax = plt.subplots(figsize=(10, 6))
cv_data = []
cv_labels = []
for name in models.keys():
    cv_data.append(cv_results[name]['scores'])
    cv_labels.append(name)

ax.boxplot(cv_data, labels=cv_labels)
ax.set_title('Cross-Validation F1 Score Distribution')
ax.set_ylabel('F1 Score (Macro)')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 6: Prediction confidence (if model supports predict_proba)
if hasattr(best_model, 'predict_proba'):
    proba = best_model.predict_proba(X_test)
    max_proba = proba.max(axis=1)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Overall confidence distribution
    axes[0].hist(max_proba, bins=20, color='#3498db', edgecolor='white')
    axes[0].set_title('Prediction Confidence Distribution')
    axes[0].set_xlabel('Max Probability')
    axes[0].set_ylabel('Count')
    axes[0].axvline(x=0.5, color='red', linestyle='--', label='50% threshold')
    axes[0].legend()
    
    # Confidence for correct vs incorrect
    correct_mask = final_predictions == y_test
    axes[1].hist(max_proba[correct_mask], bins=15, alpha=0.7, label='Correct', color='#2ecc71')
    axes[1].hist(max_proba[~correct_mask], bins=15, alpha=0.7, label='Incorrect', color='#e74c3c')
    axes[1].set_title('Confidence: Correct vs Incorrect Predictions')
    axes[1].set_xlabel('Max Probability')
    axes[1].set_ylabel('Count')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Model does not support predict_proba")

## 13. Conclusion

### Key Findings from EDA
- The dataset contains 268 expense entries from FY23 with debit amounts ranging from INR 300 to 12.7M
- Equipment purchases dominate the dataset followed by Services and Material
- Text descriptions contain strong domain-specific vocabulary that clearly differentiates categories
- Services entries tend to have longer descriptions and higher amounts

### Most Useful Features
- Domain-specific keywords (services/commissioning/pipe/cable/camera etc)
- TF-IDF bigrams capture multi-word patterns effectively
- Text length provides additional discriminative signal

### Best Model
- The best performing model achieves strong F1 scores across all categories
- TF-IDF with classical ML approach was the right choice for this small dataset
- The model generalizes well as shown by consistent CV and test performance

### Future Improvements
- Collect more labeled data to improve minority class (Material) performance
- Try ensemble of rule-based and ML approaches for production
- Consider active learning to iteratively improve labels
- Add more text features like named entity recognition for vendor/brand names

In [ ]:
# Save predictions to file
output_df = df_clean.copy()
output_df['Predicted_Category'] = le.inverse_transform(best_model.predict(X_combined))
output_df[['Year', 'Debit', 'Remarks', 'Category', 'Predicted_Category']].to_csv('predictions.csv', index=False)
print("Predictions saved to predictions.csv")
print(f"\nFinal Model: {best_model_name}")
print(f"Test Accuracy: {acc:.4f}")
print(f"Macro F1: {f1_macro:.4f}")
print(f"Weighted F1: {f1_weighted:.4f}")
print("\nDone!")